In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import torch.nn.functional as F
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import time

In [ ]:
# --- Check for GPU ---
# This will be important when you move to your cluster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#################################################################
# 1. AGENT BRAIN CLASS (As designed)
#################################################################

class AgentBrain(nn.Module):
    """
    An Actor-Critic agent with a shared RNN trunk (GRU).
    """
    def __init__(self, public_input_size, private_input_size, rnn_hidden_size, 
                 move_action_size, pull_action_size, gaze_action_size):
        
        super(AgentBrain, self).__init__()
        
        self.input_size = public_input_size + private_input_size
        self.rnn_hidden_size = rnn_hidden_size
        self.move_action_size = move_action_size
        self.pull_action_size = pull_action_size
        self.gaze_action_size = gaze_action_size

        # --- 1. The Shared RNN "Trunk" ---
        self.rnn_trunk = nn.GRUCell(self.input_size, self.rnn_hidden_size)

        # --- 2. The "Actor" Head ---
        self.move_head = nn.Linear(self.rnn_hidden_size, move_action_size)
        self.pull_head = nn.Linear(self.rnn_hidden_size, pull_action_size)
        self.gaze_head = nn.Linear(self.rnn_hidden_size, gaze_action_size)

        # --- 3. The "Critic" Head ---
        self.critic_head = nn.Linear(self.rnn_hidden_size, 1)

    def forward(self, public_input, private_gaze_input, prev_hidden_state):
        
        # --- Build the Full Input State ---
        full_input = torch.cat([public_input, private_gaze_input], dim=-1)

        # --- Run the "Accumulator" (RNN Trunk) ---
        current_hidden_state = self.rnn_trunk(full_input, prev_hidden_state)

        # --- Get Actor & Critic Outputs ---
        value = self.critic_head(current_hidden_state)
        
        move_logits = self.move_head(current_hidden_state)
        pull_logits = self.pull_head(current_hidden_state)
        gaze_logits = self.gaze_head(current_hidden_state)
        
        # Return logits, value, and the new hidden state
        return (move_logits, pull_logits, gaze_logits), value, current_hidden_state

### different environment settings

In [ ]:
#################################################################
# 2. ENVIRONMENT CLASS (Modified for Curriculum)
#################################################################

class NaturalisticCoopEnv:
    
    def __init__(self, grid_size_x=20, grid_size_y=20, session_length=20000):
        self.grid_size_x = grid_size_x
        self.grid_size_y = grid_size_y
        self.max_session_steps = session_length
        self.boundary_x = self.grid_size_x // 2 
        
        self.locations = {
            # 'lever_A': np.array([3, 18]), 'port_A':  np.array([3, 1]),
            # 'lever_B': np.array([16, 18]), 'port_B':  np.array([16, 1]),
            'lever_A': np.array([1, 8]),        # Agent A's side
            'port_A':  np.array([1, 1]),         # Agent A's side (7 steps away)
            'lever_B': np.array([6, 8]),       # Agent B's side
            'port_B':  np.array([6, 1]),        # Agent B's side (7 steps away)
        }
        
        # --- NEW: CURRICULUM PARAMETERS ---
        self.training_phase = "SELF_REWARD" # Start with self-reward
        self.current_sync_window = 10      # Final 1s window
        
        # --- Task Timers ---
        self.drink_duration_min = 30
        self.drink_duration_max = 50
        self.pull_reset_time = 5
        self.lever_cooldown_duration = 60
        
        self.reset() # Call reset to initialize all state variables

    def set_training_phase(self, phase, sync_window=10):
        """
        NEW: Public method to update the curriculum.
        phase = "SELF_REWARD" or "COOP"
        """
        self.training_phase = phase
        self.current_sync_window = sync_window
        print(f"*** ENV UPDATED: Phase={self.training_phase}, SyncWindow={self.current_sync_window} ***")

    def _get_state(self):
        state_A = {
            'self_pos': self.agent_A_pos, 'partner_pos': self.agent_B_pos,
            'lever_pos': self.locations['lever_A'], 'port_pos': self.locations['port_A'],
            'self_lever_state': self.lever_A_state, 'self_port_state': self.port_A_state,
            'is_drinking': self.agent_A_is_drinking > 0,
            'can_pull': self.agent_A_pull_cooldown == 0,
            'success_sound': self.success_sound_played 
        }
        state_B = {
            'self_pos': self.agent_B_pos, 'partner_pos': self.agent_A_pos,
            'lever_pos': self.locations['lever_B'], 'port_pos': self.locations['port_B'],
            'self_lever_state': self.lever_B_state, 'self_port_state': self.port_B_state,
            'is_drinking': self.agent_B_is_drinking > 0,
            'can_pull': self.agent_B_pull_cooldown == 0,
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        self.agent_A_pos = self.locations['port_A'].copy()
        self.agent_B_pos = self.locations['port_B'].copy()
        self.current_session_step = 0
        self.lever_A_state = 0
        self.lever_B_state = 0
        self.pull_A_timestamp = -1
        self.pull_B_timestamp = -1
        self.lever_cooldown_timer = 0
        self.port_A_state = 0
        self.port_B_state = 0
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.agent_A_is_drinking = 0
        self.agent_B_is_drinking = 0
        self.success_sound_played = 0
        return self._get_state()

    def _move(self, current_pos, move_action, agent_id):
        if move_action == -1: return current_pos
        if move_action == 0: current_pos[1] += 1
        elif move_action == 1: current_pos[1] -= 1
        elif move_action == 2: current_pos[0] -= 1
        elif move_action == 3: current_pos[0] += 1
        current_pos[1] = np.clip(current_pos[1], 0, self.grid_size_y - 1)
        if agent_id == 'A':
            current_pos[0] = np.clip(current_pos[0], 0, self.boundary_x - 1)
        elif agent_id == 'B':
            current_pos[0] = np.clip(current_pos[0], self.boundary_x, self.grid_size_x - 1)
        return current_pos

    def step(self, action_A, action_B):
        self.current_session_step += 1
        
        # Using the reward shaping from before
        reward_A, reward_B = -0.01, -0.01 
        
        self.success_sound_played = 0 

        # --- Update Timers ---
        if self.lever_cooldown_timer > 0: self.lever_cooldown_timer -= 1
        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1
        
        if self.agent_A_is_drinking > 0:
            self.agent_A_is_drinking -= 1
            action_A = {'move': -1, 'pull': 0, 'gaze': 0} 
        if self.agent_B_is_drinking > 0:
            self.agent_B_is_drinking -= 1
            action_B = {'move': -1, 'pull': 0, 'gaze': 0}
            
        # --- Process Movement ---
        if self.agent_A_is_drinking == 0:
            old_dist_A = np.linalg.norm(self.agent_A_pos - self.locations['lever_A'])
            self.agent_A_pos = self._move(self.agent_A_pos, action_A['move'], 'A')
            new_dist_A = np.linalg.norm(self.agent_A_pos - self.locations['lever_A'])
            if new_dist_A < old_dist_A: reward_A += 0.02
        if self.agent_B_is_drinking == 0:
            old_dist_B = np.linalg.norm(self.agent_B_pos - self.locations['lever_B'])
            self.agent_B_pos = self._move(self.agent_B_pos, action_B['move'], 'B')
            new_dist_B = np.linalg.norm(self.agent_B_pos - self.locations['lever_B'])
            if new_dist_B < old_dist_B: reward_B += 0.02

        # --- Update Lever Cooldown ---
        if self.lever_cooldown_timer == 0 and self.lever_A_state == 2:
            self.lever_A_state = 0 
            self.lever_B_state = 0

        # --- Process Pull Actions ---
        
        # --- CURRICULUM LOGIC ---
        if self.training_phase == "SELF_REWARD":
            # --- PHASE 1: SELF REWARD ---
            # If agent pulls, it *immediately* succeeds. No coordination needed.
            if (action_A['pull'] == 1 and self.agent_A_is_drinking == 0 and
                self.agent_A_pull_cooldown == 0 and
                np.array_equal(self.agent_A_pos, self.locations['lever_A']) and
                self.lever_A_state == 0):
                
                reward_A += 1.0 # Main reward
                self.port_A_state = 1
                self.success_sound_played = 1 # Play sound
                self.lever_A_state = 2 # Set to COOLDOWN
                self.lever_cooldown_timer = self.lever_cooldown_duration
                self.agent_A_pull_cooldown = self.pull_reset_time

            if (action_B['pull'] == 1 and self.agent_B_is_drinking == 0 and
                self.agent_B_pull_cooldown == 0 and
                np.array_equal(self.agent_B_pos, self.locations['lever_B']) and
                self.lever_B_state == 0):
                
                reward_B += 1.0 # Main reward
                self.port_B_state = 1
                self.success_sound_played = 1
                self.lever_B_state = 2
                self.lever_cooldown_timer = self.lever_cooldown_duration # (share cooldown)
                self.agent_B_pull_cooldown = self.pull_reset_time
        
        else: # self.training_phase == "COOP"
            # --- PHASE 2-4: COORDINATION ---
            # This is the original logic
            
            # Agent A pulls
            if (action_A['pull'] == 1 and self.agent_A_is_drinking == 0 and
                self.agent_A_pull_cooldown == 0 and
                np.array_equal(self.agent_A_pos, self.locations['lever_A']) and
                self.lever_A_state == 0):
                
                reward_A += 0.1 # Shaping bonus for trying
                self.lever_A_state = 1
                self.pull_A_timestamp = self.current_session_step
                self.agent_A_pull_cooldown = self.pull_reset_time

            # Agent B pulls
            if (action_B['pull'] == 1 and self.agent_B_is_drinking == 0 and
                self.agent_B_pull_cooldown == 0 and
                np.array_equal(self.agent_B_pos, self.locations['lever_B']) and
                self.lever_B_state == 0):
                
                reward_B += 0.1
                self.lever_B_state = 1
                self.pull_B_timestamp = self.current_session_step
                self.agent_B_pull_cooldown = self.pull_reset_time

            # Check for Coordination (SUCCESS)
            if self.lever_A_state == 1 and self.lever_B_state == 1:
                # Check if within the *current* sync window
                if abs(self.pull_A_timestamp - self.pull_B_timestamp) <= self.current_sync_window:
                    reward_A += 1.0 # Main reward
                    reward_B += 1.0
                    self.port_A_state = 1 
                    self.port_B_state = 1
                    self.success_sound_played = 1 
                
                self.lever_A_state = 2 
                self.lever_B_state = 2
                self.lever_cooldown_timer = self.lever_cooldown_duration
        # --- END CURRICULUM LOGIC ---

        # --- Reward Consumption (Drinking) ---
        if (self.agent_A_is_drinking == 0 and
            np.array_equal(self.agent_A_pos, self.locations['port_A']) and
            self.port_A_state == 1):
            
            reward_A += 0.5 # Bonus for cashing in
            self.port_A_state = 0 
            drink_lock_duration = np.random.randint(self.drink_duration_min, self.drink_duration_max + 1) 
            self.agent_A_is_drinking = drink_lock_duration

        if (self.agent_B_is_drinking == 0 and
            np.array_equal(self.agent_B_pos, self.locations['port_B']) and
            self.port_B_state == 1):
            
            reward_B += 0.5
            self.port_B_state = 0
            drink_lock_duration = np.random.randint(self.drink_duration_min, self.drink_duration_max + 1)
            self.agent_B_is_drinking = drink_lock_duration

        # --- Session End ---
        done = (self.current_session_step >= self.max_session_steps)
        
        next_state_A, next_state_B = self._get_state()
        
        return next_state_A, next_state_B, reward_A, reward_B, done

In [ ]:
#################################################################
# 2. ENVIRONMENT CLASS (Modified for Curriculum)
#################################################################

class Phase1A_Env:
    
    def __init__(self, grid_size_x=20, grid_size_y=20, session_length=20000):
        self.grid_size_x = grid_size_x
        self.grid_size_y = grid_size_y
        self.max_session_steps = session_length
        self.boundary_x = self.grid_size_x // 2 
        
        self.locations = {
            # 'lever_A': np.array([3, 18]), 'port_A':  np.array([3, 1]),
            # 'lever_B': np.array([16, 18]), 'port_B':  np.array([16, 1]),
            'lever_A': np.array([1, 8]),        # Agent A's side
            'port_A':  np.array([1, 1]),         # Agent A's side (7 steps away)
            'lever_B': np.array([6, 8]),       # Agent B's side
            'port_B':  np.array([6, 1]),        # Agent B's side (7 steps away)
        }
        
        # --- NEW: CURRICULUM PARAMETERS ---
        self.training_phase = "SELF_REWARD" # Start with self-reward
        self.current_sync_window = 10      # Final 1s window
        
        # --- Task Timers ---
        self.drink_duration_min = 30
        self.drink_duration_max = 50
        self.pull_reset_time = 5
        self.lever_cooldown_duration = 60
        
        self.reset() # Call reset to initialize all state variables

    def set_training_phase(self, phase, sync_window=10):
        """
        NEW: Public method to update the curriculum.
        phase = "SELF_REWARD" or "COOP"
        """
        self.training_phase = phase
        self.current_sync_window = sync_window
        print(f"*** ENV UPDATED: Phase={self.training_phase}, SyncWindow={self.current_sync_window} ***")

    def _get_state(self):
        state_A = {
            'self_pos': self.agent_A_pos, 'partner_pos': self.agent_B_pos,
            'lever_pos': self.locations['lever_A'], 'port_pos': self.locations['port_A'],
            'self_lever_state': self.lever_A_state, 'self_port_state': self.port_A_state,
            'is_drinking': self.agent_A_is_drinking > 0,
            'can_pull': self.agent_A_pull_cooldown == 0,
            'success_sound': self.success_sound_played 
        }
        state_B = {
            'self_pos': self.agent_B_pos, 'partner_pos': self.agent_A_pos,
            'lever_pos': self.locations['lever_B'], 'port_pos': self.locations['port_B'],
            'self_lever_state': self.lever_B_state, 'self_port_state': self.port_B_state,
            'is_drinking': self.agent_B_is_drinking > 0,
            'can_pull': self.agent_B_pull_cooldown == 0,
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        
        # --- PHASE 1A CHANGE: START AT THE LEVER ---
        self.agent_A_pos = self.locations['lever_A'].copy()
        self.agent_B_pos = self.locations['lever_B'].copy()
        # --- END CHANGE ---
        
        self.current_session_step = 0
        self.lever_A_state = 0
        self.lever_B_state = 0
        self.pull_A_timestamp = -1
        self.pull_B_timestamp = -1
        self.lever_cooldown_timer = 0
        self.port_A_state = 0
        self.port_B_state = 0
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.agent_A_is_drinking = 0
        self.agent_B_is_drinking = 0
        self.success_sound_played = 0
        return self._get_state()

    def _move(self, current_pos, move_action, agent_id):
        if move_action == -1: return current_pos
        if move_action == 0: current_pos[1] += 1
        elif move_action == 1: current_pos[1] -= 1
        elif move_action == 2: current_pos[0] -= 1
        elif move_action == 3: current_pos[0] += 1
        current_pos[1] = np.clip(current_pos[1], 0, self.grid_size_y - 1)
        if agent_id == 'A':
            current_pos[0] = np.clip(current_pos[0], 0, self.boundary_x - 1)
        elif agent_id == 'B':
            current_pos[0] = np.clip(current_pos[0], self.boundary_x, self.grid_size_x - 1)
        return current_pos

    def step(self, action_A, action_B):
        self.current_session_step += 1
        
        # --- NEW: Add back a small time penalty ---
        reward_A, reward_B = -0.01, -0.01
        
        self.success_sound_played = 0 

        # --- Update Timers ---
        if self.lever_cooldown_timer > 0: self.lever_cooldown_timer -= 1
        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1
        
        # --- (Drinking logic is irrelevant in this phase) ---
        
        # --- Process Movement ---
        # (No reward shaping for move, it's not needed now)
        self.agent_A_pos = self._move(self.agent_A_pos, action_A['move'], 'A')
        self.agent_B_pos = self._move(self.agent_B_pos, action_B['move'], 'B')

        # --- Update Lever Cooldown ---
        if self.lever_cooldown_timer == 0 and self.lever_A_state == 2:
            self.lever_A_state = 0 
            self.lever_B_state = 0

        # --- PHASE 1A: SELF REWARD (AT LEVER) ---
        if (action_A['pull'] == 1 and
            self.agent_A_pull_cooldown == 0 and
            np.array_equal(self.agent_A_pos, self.locations['lever_A']) and
            self.lever_A_state == 0):
            
            # --- BIG, CLEAR REWARD ---
            reward_A = 10.0 
            
            self.success_sound_played = 1 
            self.lever_A_state = 2 
            self.lever_cooldown_timer = self.lever_cooldown_duration
            self.agent_A_pull_cooldown = self.pull_reset_time

        if (action_B['pull'] == 1 and
            self.agent_B_pull_cooldown == 0 and
            np.array_equal(self.agent_B_pos, self.locations['lever_B']) and
            self.lever_B_state == 0):
            
            # --- BIG, CLEAR REWARD ---
            reward_B = 10.0
            
            self.success_sound_played = 1
            self.lever_B_state = 2
            self.lever_cooldown_timer = self.lever_cooldown_duration
            self.agent_B_pull_cooldown = self.pull_reset_time
        
        # --- 8. Check for Session End ---
        timeout = (self.current_session_step >= self.max_session_steps)
        
        # Reset on success for faster trials
        success_done = (self.lever_cooldown_timer == self.lever_cooldown_duration)
        
        done = timeout or success_done
        
        next_state_A, next_state_B = self._get_state()
        
        return next_state_A, next_state_B, reward_A, reward_B, done

In [ ]:
# --- NEW, DECOUPLED ENVIRONMENT ---
class Phase1A_Env_Decoupled:
    """
    Phase 1A Environment with DECOUPLED levers.
    Each agent's lever is on its own independent cooldown.
    """
    
    def __init__(self, grid_size_x=8, grid_size_y=10, session_length=200): # Short session
        self.grid_size_x = grid_size_x
        self.grid_size_y = grid_size_y
        self.max_session_steps = session_length # Note: this is now trial length
        self.boundary_x = self.grid_size_x // 2 
        
        self.locations = {
            'lever_A': np.array([1, 8]), 'port_A':  np.array([1, 1]),
            'lever_B': np.array([6, 8]), 'port_B':  np.array([6, 1]),
        }
        
        # --- Task Timers ---
        self.drink_duration_min = 30
        self.drink_duration_max = 50
        self.pull_reset_time = 5
        self.lever_cooldown_duration = 60 # Cooldown for a *single* lever
        
        self.training_phase = "SELF_REWARD" # This env is *only* for self-reward
        self.current_sync_window = 10      # Final 1s window
        
        self.reset() # Call reset to initialize all state variables

    def _get_state(self):
        state_A = {
            'self_pos': self.agent_A_pos, 'partner_pos': self.agent_B_pos,
            'lever_pos': self.locations['lever_A'], 'port_pos': self.locations['port_A'],
            'self_lever_state': self.lever_A_state, 'self_port_state': self.port_A_state,
            'is_drinking': self.agent_A_is_drinking > 0,
            'can_pull': self.agent_A_pull_cooldown == 0,
            'success_sound': self.success_sound_played 
        }
        state_B = {
            'self_pos': self.agent_B_pos, 'partner_pos': self.agent_A_pos,
            'lever_pos': self.locations['lever_B'], 'port_pos': self.locations['port_B'],
            'self_lever_state': self.lever_B_state, 'self_port_state': self.port_B_state,
            'is_drinking': self.agent_B_is_drinking > 0,
            'can_pull': self.agent_B_pull_cooldown == 0,
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        """ Resets the episode. """
        self.agent_A_pos = self.locations['lever_A'].copy()
        self.agent_B_pos = self.locations['lever_B'].copy()
        
        self.current_session_step = 0
        
        # --- DECOUPLED LEVER STATES ---
        self.lever_A_state = 0
        self.lever_B_state = 0
        self.lever_A_cooldown_timer = 0
        self.lever_B_cooldown_timer = 0
        # --- END DECOUPLED ---

        self.port_A_state = 0
        self.port_B_state = 0
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.agent_A_is_drinking = 0
        self.agent_B_is_drinking = 0
        self.success_sound_played = 0
        
        return self._get_state()

    def _move(self, current_pos, move_action, agent_id):
        if move_action == -1: return current_pos
        if move_action == 0: current_pos[1] += 1
        elif move_action == 1: current_pos[1] -= 1
        elif move_action == 2: current_pos[0] -= 1
        elif move_action == 3: current_pos[0] += 1
        current_pos[1] = np.clip(current_pos[1], 0, self.grid_size_y - 1)
        if agent_id == 'A':
            current_pos[0] = np.clip(current_pos[0], 0, self.boundary_x - 1)
        elif agent_id == 'B':
            current_pos[0] = np.clip(current_pos[0], self.boundary_x, self.grid_size_x - 1)
        return current_pos

    def step(self, action_A, action_B):
        self.current_session_step += 1
        reward_A, reward_B = -0.01, -0.01 
        self.success_sound_played = 0 

        # --- Update DECOUPLED Timers ---
        if self.lever_A_cooldown_timer > 0: 
            self.lever_A_cooldown_timer -= 1
            if self.lever_A_cooldown_timer == 0:
                self.lever_A_state = 0 # Lever A is READY
                
        if self.lever_B_cooldown_timer > 0: 
            self.lever_B_cooldown_timer -= 1
            if self.lever_B_cooldown_timer == 0:
                self.lever_B_state = 0 # Lever B is READY
        
        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1
        
        # (Drinking logic is not needed for this simple phase)
        self.agent_A_is_drinking = 0
        self.agent_B_is_drinking = 0
            
        # --- Process Movement ---
        self.agent_A_pos = self._move(self.agent_A_pos, action_A['move'], 'A')
        self.agent_B_pos = self._move(self.agent_B_pos, action_B['move'], 'B')

        # --- PHASE 1A: SELF REWARD (AT LEVER) ---
        if (action_A['pull'] == 1 and
            self.agent_A_pull_cooldown == 0 and
            np.array_equal(self.agent_A_pos, self.locations['lever_A']) and
            self.lever_A_state == 0): # Check *A's* lever
            
            reward_A = 10.0 
            self.success_sound_played = 1 # Play sound for both
            self.lever_A_state = 2 # Put *A's* lever on cooldown
            self.lever_A_cooldown_timer = self.lever_cooldown_duration
            self.agent_A_pull_cooldown = self.pull_reset_time

        if (action_B['pull'] == 1 and
            self.agent_B_pull_cooldown == 0 and
            np.array_equal(self.agent_B_pos, self.locations['lever_B']) and
            self.lever_B_state == 0): # Check *B's* lever
            
            reward_B = 10.0
            self.success_sound_played = 1 # Play sound for both
            self.lever_B_state = 2 # Put *B's* lever on cooldown
            self.lever_B_cooldown_timer = self.lever_cooldown_duration
            self.agent_B_pull_cooldown = self.pull_reset_time
        
        # --- 8. Check for Trial End ---
        timeout = (self.current_session_step >= self.max_session_steps)
        done = timeout
        # --- END NEW LOGIC ---
        
        next_state_A, next_state_B = self._get_state()
        
        return next_state_A, next_state_B, reward_A, reward_B, done

In [ ]:
#################################################################
# 2. MASTER CURRICULUM ENVIRONMENT (v7 - Medium Penalty -0.5)
#################################################################
class CurriculumEnv:
    """
    Continuous, non-episodic environment (v7 - Medium Penalty)
    - 'done' is ALWAYS False.
    - 'reset()' is only called once.
    - GOAL: Test if a "medium" -0.5 penalty can fix the "spam" policy.
    
    - --- NEW REWARDS ---
    - +1.0 "Social Gaze" reward (from Run 4)
    - -0.5 "Medium Penalty" for pulling on cooldown (NEW)
    """
    
    def __init__(self, grid_size_x=8, grid_size_y=10):
        # (Same as before)
        self.grid_size_x = grid_size_x
        self.grid_size_y = grid_size_y
        self.boundary_x = self.grid_size_x // 2 
        self.locations = {
            'lever_A': np.array([1, 8]), 'port_A':  np.array([1, 1]),
            'lever_B': np.array([6, 8]), 'port_B':  np.array([6, 1]),
        }
        self.drink_duration_min = 30
        self.drink_duration_max = 50
        self.pull_reset_time = 5
        self.lever_cooldown_duration = 60 
        self.training_phase = "SELF_REWARD" 
        self.current_sync_window = 30 
        self.reset() 

    def set_training_phase(self, phase, sync_window=30):
        # (Same as before)
        self.training_phase = phase
        self.current_sync_window = sync_window
        print(f"*** ENV UPDATED: Phase={self.training_phase}, SyncWindow={self.current_sync_window} ***")

    def _get_state(self):
        # --- This is the "boolean" can_pull state ---
        lever_A_state = 0 # 0=Ready
        if self.training_phase == "COOP_FULL":
            if self.shared_cooldown_timer > 0: lever_A_state = 2 # 2=Cooldown
        else: # SELF_REWARD or COOP_ASSISTED (decoupled)
            if self.lever_A_cooldown_timer > 0: lever_A_state = 2
            
        lever_B_state = 0
        if self.training_phase == "COOP_FULL":
            if self.shared_cooldown_timer > 0: lever_B_state = 2
        else: # SELF_REWARD or COOP_ASSISTED (decoupled)
            if self.lever_B_cooldown_timer > 0: lever_B_state = 2
            
        state_A = {
            'self_pos': self.agent_A_pos,
            'lever_pos': self.locations['lever_A'],
            'port_pos': self.locations['port_A'],
            'self_lever_state': lever_A_state,
            'self_port_state': self.port_A_state,
            'is_drinking': self.agent_A_is_drinking > 0,
            'can_pull': self.agent_A_pull_cooldown == 0, # <-- The boolean signal
            'success_sound': self.success_sound_played 
        }
        state_B = {
            'self_pos': self.agent_B_pos,
            'lever_pos': self.locations['lever_B'],
            'port_pos': self.locations['port_B'],
            'self_lever_state': lever_B_state,
            'self_port_state': self.port_B_state,
            'is_drinking': self.agent_B_is_drinking > 0,
            'can_pull': self.agent_B_pull_cooldown == 0, # <-- The boolean signal
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        # (Same as before)
        self.agent_A_pos = self.locations['port_A'].copy()
        self.agent_B_pos = self.locations['port_B'].copy()
        self.current_step = 0 
        self.lever_A_state = 0
        self.lever_B_state = 0
        self.lever_A_cooldown_timer = 0
        self.lever_B_cooldown_timer = 0
        self.pull_A_timestamp = -1 
        self.pull_B_timestamp = -1 
        self.shared_cooldown_timer = 0 
        self.port_A_state = 0
        self.port_B_state = 0
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.agent_A_is_drinking = 0
        self.agent_B_is_drinking = 0
        self.success_sound_played = 0
        return self._get_state()

    def _move(self, current_pos, move_action, agent_id):
        # (Same as before)
        if move_action == -1: return current_pos
        if move_action == 0: current_pos[1] += 1
        elif move_action == 1: current_pos[1] -= 1
        elif move_action == 2: current_pos[0] -= 1
        elif move_action == 3: current_pos[0] += 1
        current_pos[1] = np.clip(current_pos[1], 0, self.grid_size_y - 1)
        if agent_id == 'A':
            current_pos[0] = np.clip(current_pos[0], 0, self.boundary_x - 1)
        elif agent_id == 'B':
            current_pos[0] = np.clip(current_pos[0], self.boundary_x, self.grid_size_x - 1)
        return current_pos

    def step(self, action_A, action_B):
        self.current_step += 1
        reward_A, reward_B = -0.01, -0.01 
        self.success_sound_played = 0 
        done = False
        coordination_attempt_complete = False 

        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1
        
        if self.agent_A_is_drinking > 0:
            self.agent_A_is_drinking -= 1
            action_A = {'move': -1, 'pull': 0, 'gaze': 0} 
        if self.agent_B_is_drinking > 0:
            self.agent_B_is_drinking -= 1
            action_B = {'move': -1, 'pull': 0, 'gaze': 0}

        if self.agent_A_is_drinking == 0:
            self.agent_A_pos = self._move(self.agent_A_pos, action_A['move'], 'A')
        if self.agent_B_is_drinking == 0:
            self.agent_B_pos = self._move(self.agent_B_pos, action_B['move'], 'B')

        at_lever_A = np.array_equal(self.agent_A_pos, self.locations['lever_A'])
        at_lever_B = np.array_equal(self.agent_B_pos, self.locations['lever_B'])

        if self.training_phase == "SELF_REWARD":
            # --- PHASE 1: Decoupled Self-Reward Loop ---
            if self.lever_A_cooldown_timer > 0: 
                self.lever_A_cooldown_timer -= 1
            if self.lever_B_cooldown_timer > 0: 
                self.lever_B_cooldown_timer -= 1
            
            # --- Agent A Pull Logic ---
            if (action_A['pull'] == 1 and self.agent_A_is_drinking == 0 and at_lever_A):
                if self.agent_A_pull_cooldown > 0 or self.lever_A_cooldown_timer > 0:
                    # --- "MEDIUM" PENALTY ---
                    reward_A -= 0.5 
                elif self.agent_A_pull_cooldown == 0 and self.lever_A_cooldown_timer == 0:
                    # --- SUCCESS ---
                    reward_A += 10.0 
                    self.port_A_state = 1 
                    self.success_sound_played = 1
                    self.lever_A_cooldown_timer = self.lever_cooldown_duration
                    self.agent_A_pull_cooldown = self.pull_reset_time

            # --- Agent B Pull Logic ---
            if (action_B['pull'] == 1 and self.agent_B_is_drinking == 0 and at_lever_B):
                if self.agent_B_pull_cooldown > 0 or self.lever_B_cooldown_timer > 0:
                    # --- "MEDIUM" PENALTY ---
                    reward_B -= 0.5 
                elif self.agent_B_pull_cooldown == 0 and self.lever_B_cooldown_timer == 0:
                    # --- SUCCESS ---
                    reward_B += 10.0
                    self.port_B_state = 1
                    self.success_sound_played = 1
                    self.lever_B_cooldown_timer = self.lever_cooldown_duration
                    self.agent_B_pull_cooldown = self.pull_reset_time

        else: # "COOP_ASSISTED" or "COOP_FULL"
            # --- PHASE 2-6: Coordinated Pull ---
            
            # --- (BOOSTED) "SOCIAL GAZE" BREADCRUMB ---
            if action_A['gaze'] == 1 and at_lever_B:
                reward_A += 1.0 # Keep this high
            if action_B['gaze'] == 1 and at_lever_A:
                reward_B += 1.0 # Keep this high
                
            if at_lever_A and at_lever_B:
                reward_A += 1.0 # Keep this high
                reward_B += 1.0 # Keep this high
            # --- END BREADCRUMB ---

            if self.shared_cooldown_timer > 0: 
                self.shared_cooldown_timer -= 1
                
                # --- "MEDIUM" PENALTY ---
                if action_A['pull'] == 1 and at_lever_A and self.agent_A_is_drinking == 0:
                    reward_A -= 0.5
                if action_B['pull'] == 1 and at_lever_B and self.agent_B_is_drinking == 0:
                    reward_B -= 0.5
                
            else: # Lever is READY
                
                # --- Agent A pull attempt ---
                if (action_A['pull'] == 1 and self.agent_A_is_drinking == 0 and at_lever_A):
                    if self.agent_A_pull_cooldown > 0:
                        # --- "MEDIUM" PENALTY ---
                        reward_A -= 0.5
                    else:
                        # --- This is a VALID pull attempt ---
                        self.pull_A_timestamp = self.current_step
                        self.agent_A_pull_cooldown = self.pull_reset_time
                        if self.pull_B_timestamp != -1: 
                            coordination_attempt_complete = True
                
                # --- Agent B pull attempt ---
                if (action_B['pull'] == 1 and self.agent_B_is_drinking == 0 and at_lever_B):
                    if self.agent_B_pull_cooldown > 0:
                        # --- "MEDIUM" PENALTY ---
                        reward_B -= 0.5
                    else:
                        # --- This is a VALID pull attempt ---
                        self.pull_B_timestamp = self.current_step
                        self.agent_B_pull_cooldown = self.pull_reset_time
                        if self.pull_A_timestamp != -1: 
                            coordination_attempt_complete = True
            
            # --- Check for Coordination ---
            if coordination_attempt_complete:
                if abs(self.pull_A_timestamp - self.pull_B_timestamp) <= self.current_sync_window:
                    # SUCCESS
                    if self.training_phase == "COOP_ASSISTED":
                        reward_A += 20.0; reward_B += 20.0
                    else: # COOP_FULL
                        reward_A += 10.0; reward_B += 10.0
                    self.port_A_state = 1; self.port_B_state = 1 
                    self.success_sound_played = 1 
                
                else:
                    # FAILED COORD
                    if self.training_phase == "COOP_ASSISTED":
                        reward_A += 10.0; reward_B += 10.0
                        self.port_A_state = 1; self.port_B_state = 1
                    else: # COOP_FULL
                        reward_A -= 0.5
                        reward_B -= 0.5
                
                # Reset for next attempt
                self.shared_cooldown_timer = self.lever_cooldown_duration
                self.pull_A_timestamp = -1
                self.pull_B_timestamp = -1

        # --- Reward Consumption (Drinking) ---
        if (self.agent_A_is_drinking == 0 and
            np.array_equal(self.agent_A_pos, self.locations['port_A']) and
            self.port_A_state == 1):
            
            reward_A += 15.0 
            self.port_A_state = 0 
            drink_lock = np.random.randint(self.drink_duration_min, self.drink_duration_max + 1) 
            self.agent_A_is_drinking = drink_lock
            
        if (self.agent_B_is_drinking == 0 and
            np.array_equal(self.agent_B_pos, self.locations['port_B']) and
            self.port_B_state == 1):
            
            reward_B += 15.0
            self.port_B_state = 0
            drink_lock = np.random.randint(self.drink_duration_min, self.drink_duration_max + 1)
            self.agent_B_is_drinking = drink_lock
        
        next_state_A, next_state_B = self._get_state()
        
        return next_state_A, next_state_B, reward_A, reward_B, done

### helper functions

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS (Winning Setup - Run 4)
#################################################################

def preprocess_state(state_dict, grid_size_x, grid_size_y):
    """
    Converts the public state dictionary into a flat tensor.
    --- USES 'can_pull' BOOLEAN AND EXCLUDES PARTNER'S POSITION ---
    """
    self_pos = state_dict['self_pos'] / np.array([grid_size_x, grid_size_y])
    # partner_pos = ... # <-- REMOVED
    lever_pos = state_dict['lever_pos'] / np.array([grid_size_x, grid_size_y])
    port_pos = state_dict['port_pos'] / np.array([grid_size_x, grid_size_y])
    
    lever_state = F.one_hot(torch.tensor(state_dict['self_lever_state']), num_classes=3).float()
    port_state = F.one_hot(torch.tensor(state_dict['self_port_state']), num_classes=2).float()
    
    is_drinking = torch.tensor([1.0 if state_dict['is_drinking'] else 0.0])
    
    # --- This is the correct boolean signal ---
    can_pull = torch.tensor([1.0 if state_dict['can_pull'] else 0.0])
    
    success_sound = torch.tensor([1.0 if state_dict['success_sound'] else 0.0])
    
    tensor_state = torch.cat([
        torch.from_numpy(self_pos).float(),
        # torch.from_numpy(partner_pos).float(), # <-- REMOVED
        torch.from_numpy(lever_pos).float(),
        torch.from_numpy(port_pos).float(),
        lever_state,
        port_state,
        is_drinking,
        can_pull,
        success_sound
    ], dim=0)
    
    return tensor_state.unsqueeze(0).to(device) # Add batch dim

def preprocess_partner_pos(partner_pos_array, grid_size_x, grid_size_y):
    """ NEW: Helper to normalize the partner's position for the private input """
    pos_tensor = torch.from_numpy(partner_pos_array).float()
    pos_normalized = pos_tensor / torch.tensor([grid_size_x, grid_size_y]).float()
    return pos_normalized.unsqueeze(0).to(device) # Add batch dim


def sample_actions(logits_tuple):
    # (This function is the same)
    move_logits, pull_logits, gaze_logits = logits_tuple
    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample()
    move_log_prob = move_dist.log_prob(move_action)
    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    pull_log_prob = pull_dist.log_prob(pull_action)
    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    gaze_log_prob = gaze_dist.log_prob(gaze_action)
    total_log_prob = move_log_prob + pull_log_prob + gaze_log_prob
    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    return actions_dict, total_log_prob

### main training function

In [ ]:
#################################################################
# 4. TRAINING HYPERPARAMETERS (NEW)
#################################################################

# --- Environment ---
GRID_X = 8 # 20
GRID_Y = 10 # 20
SESSION_LENGTH = 200 # Short session for prototyping
EPISODE_LENGTH = 200     # Max steps per trial

# --- Model ---
RNN_HIDDEN_SIZE = 128 # 64

# --- NEW INPUT SIZES ---
# 2(pos) + 2(lever) + 2(port) + 3(lever_state) + 2(port_state) + 1(drink) + 1(pull) + 1(sound) = 14
PUBLIC_INPUT_SIZE = 14 
# Partner's hidden state (64) + Partner's position (2) = 66
PRIVATE_INPUT_SIZE = RNN_HIDDEN_SIZE + 2

MOVE_ACTIONS = 4  # U, D, L, R
PULL_ACTIONS = 2  # No, Yes
GAZE_ACTIONS = 2  # No, Yes

# --- Training ---
LR = 1e-4                # Learning rate
GAMMA = 0.99             # Discount factor for future rewards
N_STEPS = 500            # (WAS 500) Shorter rollout for faster updates
TOTAL_TRAIN_STEPS = 1400000  # (WAS 1.2M) Extended for the final run
ENTROPY_COEF = 0.0001     # (WAS 0.0001) Encourage exploration!
VALUE_LOSS_COEF = 1.0    # (WAS 0.5) Force the Critic to learn faster

In [ ]:
#################################################################
# 5. THE MASTER TRAINING LOOP (UPDATED)
#################################################################
def train():
    print("Starting training with FULL 6-PHASE CURRICULUM (Gaze-Gated Pos)...")
    
    env = CurriculumEnv(GRID_X, GRID_Y)
    
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR)

    # --- NEW 8-PHASE CURRICULUM TIMINGS ---
    # --- FINAL 1.4M STEP CURRICULUM TIMINGS ---
    CURRICULUM_TIMINGS = {
        0:        ("SELF_REWARD", 200),      # Phase 1 (0-200k)
        200000:   ("COOP_ASSISTED", 200),    # Phase 2 (200k-400k)
        400000:   ("COOP_FULL", 200),        # Phase 3 (400k-500k)
        500000:   ("COOP_FULL", 100),        # Phase 4 (500k-600k)
        
        # --- Give 200k steps for EACH hard window ---
        600000:   ("COOP_FULL", 50),         # Phase 5 (600k-800k)
        800000:   ("COOP_FULL", 30),         # Phase 6 (800k-1M) 
        1000000:  ("COOP_FULL", 20),         # Phase 7 (1M - 1.2M)
        1200000:  ("COOP_FULL", 10)          # Phase 8 (1.2M - 1.4M)
    }
    
    # --- Get Initial States (Called ONCE) ---
    env.set_training_phase("SELF_REWARD", sync_window=200)
    state_A_dict, state_B_dict = env.reset()
    
    public_A = preprocess_state(state_A_dict, GRID_X, GRID_Y)
    public_B = preprocess_state(state_B_dict, GRID_X, GRID_Y)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

    # --- NEW: Initialize the log ---
    full_training_log = []
    
    for update_step in range(TOTAL_TRAIN_STEPS // N_STEPS):
        
        total_steps_so_far = update_step * N_STEPS
        
        # --- Check for curriculum update ---
        for milestone_step in sorted(CURRICULUM_TIMINGS.keys(), reverse=True):
            if total_steps_so_far >= milestone_step:
                phase, window = CURRICULUM_TIMINGS[milestone_step]
                if env.training_phase != phase or env.current_sync_window != window:
                    env.set_training_phase(phase, sync_window=window)
                break 
        
        # --- 1. Rollout Phase (Collect N_STEPS) ---
        all_log_probs_A, all_values_A, all_rewards_A = [], [], []
        all_log_probs_B, all_values_B, all_rewards_B = [], [], []
        all_entropy_A, all_entropy_B = 0, 0

        hidden_A = hidden_A.detach()
        hidden_B = hidden_B.detach()

        for step_in_rollout in range(N_STEPS): # <-- Added 'step_in_rollout'
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A)
            actions_A_dict, log_prob_A = sample_actions(logits_A_tuple)
            entropy_A = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_A_tuple)
            all_entropy_A += entropy_A
            
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B)
            actions_B_dict, log_prob_B = sample_actions(logits_B_tuple)
            entropy_B = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_B_tuple)
            all_entropy_B += entropy_B

            # --- *** 1. MOVED THIS LINE UP *** ---
            # Get the result from the environment *before* logging
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)
                
            # --- LOG THE DATA ---
            # We log the state (position) *before* the action,
            # and the action that was chosen in that state.
            log_step = total_steps_so_far + step_in_rollout
            full_training_log.append({
                'step': log_step,
                'phase': env.training_phase,
                'window': env.current_sync_window,
                'A_pos_x': env.agent_A_pos[0],
                'A_pos_y': env.agent_A_pos[1],
                'A_action_move': actions_A_dict['move'],
                'A_action_pull': actions_A_dict['pull'],
                'A_action_gaze': actions_A_dict['gaze'],
                'A_reward': reward_A, # <-- *** 2. ADDED THIS LINE ***
                'B_pos_x': env.agent_B_pos[0],
                'B_pos_y': env.agent_B_pos[1],
                'B_action_move': actions_B_dict['move'],
                'B_action_pull': actions_B_dict['pull'],
                'B_action_gaze': actions_B_dict['gaze'],
                'B_reward': reward_B, # <-- *** 2. ADDED THIS LINE ***
            })
            # --- END LOGGING ---


            all_log_probs_A.append(log_prob_A)
            all_values_A.append(value_A)
            all_rewards_A.append(reward_A)
            all_log_probs_B.append(log_prob_B)
            all_values_B.append(value_B)
            all_rewards_B.append(reward_B)

            partner_pos_B_tensor = preprocess_partner_pos(env.agent_B_pos, GRID_X, GRID_Y)
            partner_pos_A_tensor = preprocess_partner_pos(env.agent_A_pos, GRID_X, GRID_Y)
            
            if actions_A_dict['gaze'] == 1:
                private_A = torch.cat([next_hidden_B.detach(), partner_pos_B_tensor], dim=-1)
            else:
                private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
            if actions_B_dict['gaze'] == 1:
                private_B = torch.cat([next_hidden_A.detach(), partner_pos_A_tensor], dim=-1)
            else:
                private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

            public_A = preprocess_state(next_state_A_dict, GRID_X, GRID_Y)
            public_B = preprocess_state(next_state_B_dict, GRID_X, GRID_Y)
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B

        # --- 2. Learning Phase (Calculate Loss & Backprop) ---
        _, next_value_A, _ = agent_A(public_A, private_A, hidden_A)
        _, next_value_B, _ = agent_B(public_B, private_B, hidden_B)

        
        
        # (This section is identical)
        rewards_A = torch.tensor(all_rewards_A).float().to(device)
        values_A = torch.cat(all_values_A)
        log_probs_A = torch.cat(all_log_probs_A)
        returns_A = torch.zeros_like(rewards_A)
        R_A = next_value_A.detach()
        for t in reversed(range(len(rewards_A))):
            R_A = rewards_A[t] + GAMMA * R_A
            returns_A[t] = R_A
        advantages_A = returns_A - values_A.detach() 
        actor_loss_A = -(log_probs_A * advantages_A).mean()
        critic_loss_A = F.mse_loss(values_A.squeeze(), returns_A)
        loss_A = (actor_loss_A + VALUE_LOSS_COEF * critic_loss_A - ENTROPY_COEF * all_entropy_A)

        rewards_B = torch.tensor(all_rewards_B).float().to(device)
        values_B = torch.cat(all_values_B)
        log_probs_B = torch.cat(all_log_probs_B)
        returns_B = torch.zeros_like(rewards_B)
        R_B = next_value_B.detach()
        for t in reversed(range(len(rewards_B))):
            R_B = rewards_B[t] + GAMMA * R_B
            returns_B[t] = R_B
        advantages_B = returns_B - values_B.detach()
        actor_loss_B = -(log_probs_B * advantages_B).mean()
        critic_loss_B = F.mse_loss(values_B.squeeze(), returns_B)
        loss_B = (actor_loss_B + VALUE_LOSS_COEF * critic_loss_B - ENTROPY_COEF * all_entropy_B)

        # --- 3. Optimization Step ---
        # (This section is identical)
        optimizer_A.zero_grad()
        loss_A.backward()
        torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 1.0)
        optimizer_A.step()
        optimizer_B.zero_grad()
        loss_B.backward()
        torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 1.0)
        optimizer_B.step()
        
        if update_step % (5000 // N_STEPS) == 0: 
            total_steps_so_far = update_step * N_STEPS
            total_rewards_A = sum(all_rewards_A)
            total_rewards_B = sum(all_rewards_B)
            print(f"Step: {total_steps_so_far}, Phase: {env.training_phase}, Win: {env.current_sync_window}, "
                  f"R_A: {total_rewards_A:.2f}, R_B: {total_rewards_B:.2f}, "
                  f"Loss_A: {loss_A.item():.4f}, Loss_B: {loss_B.item():.4f}")

    print("Training complete.")
    
    # --- NEW: Save the log data at the end ---
    print("Converting log to DataFrame...")
    df = pd.DataFrame(full_training_log)
    df.to_pickle("full_training_log_old.pkl")
    print("Full training log saved to 'full_training_log_old.pkl'")
    # --- END NEW ---
    
    torch.save(agent_A.state_dict(), "agent_A_brain_FINAL_old.pth")
    torch.save(agent_B.state_dict(), "agent_B_brain_FINAL_old.pth")
    

# --- To run the training ---

# training or not
if 1:
    if __name__ == "__main__":
        # You will need to paste the helper functions here
        # (preprocess_state, preprocess_partner_pos, sample_actions)
        # and the AgentBrain class definition
        train()
#
else:
    print('no training, will load existing trained data')

In [ ]:
# load and analysis the data from the training process
df_training_log = pd.read_pickle("full_training_log_old.pkl")
df = df_training_log


In [ ]:
df

In [ ]:
print("Calculating pull efficiency...")
import seaborn as sns


# Use the 100-step bin as requested
bin_size = 100
df['step_bin'] = (df['step'] // bin_size) * bin_size

# Define "attempted" vs "successful" pulls
# A "successful" pull is one that resulted in a large reward (> 5.0)
df['A_attempted_pull'] = (df['A_action_pull'] == 1)
df['B_attempted_pull'] = (df['B_action_pull'] == 1)
df['A_successful_pull'] = (df['A_action_pull'] == 1) & (df['A_reward'] > 5.0)
df['B_successful_pull'] = (df['B_action_pull'] == 1) & (df['B_reward'] > 5.0)


# --- 3. Aggregate Data ---
# Group by the 100-step bins and sum up the attempts and successes
df_agg = df.groupby('step_bin').agg(
    A_attempted_pulls=('A_attempted_pull', 'sum'),
    A_successful_pulls=('A_successful_pull', 'sum'),
    B_attempted_pulls=('B_attempted_pull', 'sum'),
    B_successful_pulls=('B_successful_pull', 'sum'),
    phase=('phase', 'first'),
    window=('window', 'first')
).reset_index()

# --- 4. Calculate Success Rate (Pull Efficiency) ---
# Calculate the raw success rate for each bin
# We use .fillna(0) to handle bins with 0 attempts (0/0 = NaN)
df_agg['A_success_rate'] = (df_agg['A_successful_pulls'] / df_agg['A_attempted_pulls']).fillna(0)
df_agg['B_success_rate'] = (df_agg['B_successful_pulls'] / df_agg['B_attempted_pulls']).fillna(0)

# --- 5. Smooth the Data (Rolling Average) ---
# The raw 100-step rate is too noisy. Let's smooth it with a 10,000-step
# (100 bins) rolling average to see the trend.
rolling_window_size = 100 
df_agg['A_rate_rolling'] = df_agg['A_success_rate'].rolling(window=rolling_window_size, min_periods=1).mean()
df_agg['B_rate_rolling'] = df_agg['B_success_rate'].rolling(window=rolling_window_size, min_periods=1).mean()

print("Calculation complete. Plotting...")

# --- 6. Plotting ---
plt.figure(figsize=(20, 10))
sns.lineplot(data=df_agg, x='step_bin', y='A_rate_rolling', label='Agent A: Pull Efficiency (10k-step avg)')
sns.lineplot(data=df_agg, x='step_bin', y='B_rate_rolling', label='Agent B: Pull Efficiency (10k-step avg)')

plt.title('Pull Efficiency Over Training (Successful Pulls / Total Attempted Pulls)', fontsize=18)
plt.ylabel('Pull Efficiency (Rolling Average)', fontsize=14)
plt.xlabel('Training Step', fontsize=14)
plt.legend(loc='upper left', fontsize=12)

# --- Add vertical lines for curriculum changes ---
phase_changes = df_agg.drop_duplicates('phase')[['step_bin', 'phase']].iloc[1:]
window_changes = df_agg.drop_duplicates('window')[['step_bin', 'window']].iloc[1:]

ax = plt.gca()
for _, row in phase_changes.iterrows():
    ax.axvline(x=row['step_bin'], color='red', linestyle='--', linewidth=2)
    ax.text(row['step_bin'] + 5000, 0.095, f"Phase: {row['phase']}", 
            color='red', rotation=90, ha='left', va='top', fontsize=12)

for _, row in window_changes.iterrows():
    ax.axvline(x=row['step_bin'], color='blue', linestyle=':', linewidth=2)
    ax.text(row['step_bin'] + 5000, 0.07, f"Window: {row['window']}", 
            color='blue', rotation=90, ha='left', va='top', fontsize=12)

plt.ylim(0, 0.1) # Efficiency is a rate from 0 to 1
plt.savefig("pull_efficiency_plot.pdf")

In [ ]:
# Set plotting style
sns.set(style="darkgrid")
plt.rcParams['figure.figsize'] = (18, 8)


# --- 2. Get Curriculum Change Points ---
# We do this first, on the full dataframe
bin_size = 5000
df['step_bin'] = (df['step'] // bin_size) * bin_size
df_agg_bins = df.groupby('step_bin').first().reset_index()
phase_changes = df_agg_bins.drop_duplicates('phase')[['step', 'phase']].iloc[1:]
window_changes = df_agg_bins.drop_duplicates('window')[['step', 'window']].iloc[1:]


# --- 3. Feature Engineering: Find Successful Pulls ---
print("Calculating Inter-Pull Intervals (IPI)...")

# A "successful" pull is one that resulted in a large reward (> 5.0)
df['A_successful_pull'] = (df['A_action_pull'] == 1) & (df['A_reward'] > 5.0)
df['B_successful_pull'] = (df['B_action_pull'] == 1) & (df['B_reward'] > 5.0)

# --- 4. Calculate IPI ---
# Create new, "sparse" DataFrames containing only the successful pulls
df_A_pulls = df[df['A_successful_pull']].copy()
df_B_pulls = df[df['B_successful_pull']].copy()

# Calculate the difference (in steps) between each successful pull
df_A_pulls['A_IPI'] = df_A_pulls['step'].diff()
df_B_pulls['B_IPI'] = df_B_pulls['step'].diff()

# --- 5. Smooth the Data (Rolling Average) ---
# Calculate the rolling average of the last 50 IPIs to see the trend
rolling_window_pulls = 50
df_A_pulls['A_IPI_rolling'] = df_A_pulls['A_IPI'].rolling(window=rolling_window_pulls, min_periods=1).mean()
df_B_pulls['B_IPI_rolling'] = df_B_pulls['B_IPI'].rolling(window=rolling_window_pulls, min_periods=1).mean()

print("Calculation complete. Plotting...")

# --- 6. Plotting ---
plt.figure(figsize=(20, 10))
sns.lineplot(data=df_A_pulls, x='step', y='A_IPI_rolling', label=f'Agent A: Avg. IPI (last {rolling_window_pulls} pulls)')
sns.lineplot(data=df_B_pulls, x='step', y='B_IPI_rolling', label=f'Agent B: Avg. IPI (last {rolling_window_pulls} pulls)')

# Add a line for the "optimal" pull interval (60-step cooldown + 5-step personal)
plt.axhline(y=65, color='gray', linestyle=':', linewidth=2, label='Optimal IPI (65 steps)')

plt.title('Inter-Pull Interval (IPI) Over Training', fontsize=18)
plt.ylabel('Average Interval (steps)', fontsize=14)
plt.xlabel('Training Step', fontsize=14)
plt.legend(loc='upper left', fontsize=12)
plt.ylim(0, 1200) # Zoom in on a reasonable IPI range

# --- Add vertical lines for curriculum changes ---
ax = plt.gca()
for _, row in phase_changes.iterrows():
    ax.axvline(x=row['step'], color='red', linestyle='--', linewidth=2)
    ax.text(row['step'] + 5000, 290, f"Phase: {row['phase']}", 
            color='red', rotation=90, ha='left', va='top', fontsize=12)

for _, row in window_changes.iterrows():
    ax.axvline(x=row['step'], color='blue', linestyle=':', linewidth=2)
    ax.text(row['step'] + 5000, 240, f"Window: {row['window']}", 
            color='blue', rotation=90, ha='left', va='top', fontsize=12)

plt.savefig("ipi_plot.pdf")
print("Plot saved as 'ipi_plot.pdf")

## use trained data to test the hypothesis
### helper functions

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS (NOW INCLUDES STOCHASTIC SAMPLING)
#################################################################

def sample_actions(logits_tuple):
    """
    Takes logits from the Actor and STOCHASTICALLY samples
    one action from each category.
    """
    move_logits, pull_logits, gaze_logits = logits_tuple
    
    # --- Move Action ---
    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample() # Sampled action (e.g., tensor(2))
    
    # --- Pull Action ---
    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    
    # --- Gaze Action ---
    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    
    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    
    # We don't need log_probs for analysis
    return actions_dict

### main testing functions

In [ ]:
def collect_data(num_steps_to_collect=10000):
    """
    Main function to load trained models and collect data.
    --- NOW USES STOCHASTIC SAMPLING ---
    """
    print(f"Loading models and setting up env for STOCHASTIC analysis...")
    
    # --- 1. Set up Environment ---
    env = CurriculumEnv(GRID_X, GRID_Y)
    env.set_training_phase("COOP_FULL", sync_window=10) # Set to final, 1s phase
    
    # --- 2. Load Trained Agents ---
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    agent_A.load_state_dict(torch.load("agent_A_brain_FINAL.pth"))
    agent_B.load_state_dict(torch.load("agent_B_brain_FINAL.pth"))
    
    agent_A.eval()
    agent_B.eval()

    # --- 3. Initialize States ---
    state_A_dict, state_B_dict = env.reset()
    
    public_A = preprocess_state(state_A_dict, GRID_X, GRID_Y)
    public_B = preprocess_state(state_B_dict, GRID_X, GRID_Y)
    
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

    # --- 4. Data Collection Loop ---
    analysis_data = [] 
    
    print(f"Running for {num_steps_to_collect} steps...")
    
    for step in range(num_steps_to_collect):
        if step % 1000 == 0:
            print(f"Step {step}/{num_steps_to_collect}...")
            
        with torch.no_grad():
            
            # --- Agent A runs ---
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A)
            
            # --- THIS IS THE CHANGE ---
            actions_A_dict = sample_actions(logits_A_tuple) # Was get_deterministic_actions
            # --- END CHANGE ---
            
            # --- Agent B runs ---
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B)
            
            # --- THIS IS THE CHANGE ---
            actions_B_dict = sample_actions(logits_B_tuple) # Was get_deterministic_actions
            # --- END CHANGE ---

            # --- Environment steps ---
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)

            # --- Store Data ---
            analysis_data.append({
                'step': step,
                'A_pos_x': state_A_dict['self_pos'][0],
                'A_pos_y': state_A_dict['self_pos'][1],
                'A_hidden_state': hidden_A.detach().cpu().numpy().flatten(),
                'A_value': value_A.item(),
                'A_action_move': actions_A_dict['move'],
                'A_action_pull': actions_A_dict['pull'],
                'A_action_gaze': actions_A_dict['gaze'],
                'A_reward': reward_A,
                'A_is_drinking': state_A_dict['is_drinking'],
                'B_pos_x': state_B_dict['self_pos'][0],
                'B_pos_y': state_B_dict['self_pos'][1],
                'B_hidden_state': hidden_B.detach().cpu().numpy().flatten(),
                'B_value': value_B.item(),
                'B_action_move': actions_B_dict['move'],
                'B_action_pull': actions_B_dict['pull'],
                'B_action_gaze': actions_B_dict['gaze'],
                'B_reward': reward_B,
                'B_is_drinking': state_B_dict['is_drinking'],
                'lever_A_state': state_A_dict['self_lever_state'],
                'lever_B_state': state_B_dict['self_lever_state'],
                'port_A_state': state_A_dict['self_port_state'],
                'port_B_state': state_B_dict['self_port_state'],
                'success_sound': state_A_dict['success_sound']
            })

            # --- Prepare inputs for next step ---
            partner_pos_B_tensor = preprocess_partner_pos(env.agent_B_pos, GRID_X, GRID_Y)
            partner_pos_A_tensor = preprocess_partner_pos(env.agent_A_pos, GRID_X, GRID_Y)
            
            if actions_A_dict['gaze'] == 1:
                private_A = torch.cat([next_hidden_B.detach(), partner_pos_B_tensor], dim=-1)
            else:
                private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
            if actions_B_dict['gaze'] == 1:
                private_B = torch.cat([next_hidden_A.detach(), partner_pos_A_tensor], dim=-1)
            else:
                private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

            public_A = preprocess_state(next_state_A_dict, GRID_X, GRID_Y)
            public_B = preprocess_state(next_state_B_dict, GRID_X, GRID_Y)
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B
            
            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict
            
    print("...Data collection complete.")
    return analysis_data


# --- To run the analysis ---

if 0:

    if __name__ == "__main__":

        # 1. Run the collection
        collected_data = collect_data(num_steps_to_collect=10000)

        # 2. Convert to Pandas DataFrame for analysis
        df = pd.DataFrame(collected_data)

        print(f"\nSuccessfully collected {len(df)} steps of data.")
        print("DataFrame head:")
        print(df.head())

        # 3. Save to a file (optional, but recommended)
        df.to_pickle("analysis_data.pkl")
        print("\nData saved to 'analysis_data.pkl'")

else:
    print('no testing')

In [ ]:
plt.plot(df['A_reward'])
plt.plot(df['B_reward'])


In [ ]:
plt.plot(df['A_pos_y'])